# Out-of-Sample Validation Against OSHA's Own Coding (OIICS)

Tests the two narrative classification rules in this repository, resident handling ([`../nursing-facility-handling/`](../nursing-facility-handling/)) and violence ([`../resident-violence/`](../resident-violence/)), against the injury codes OSHA attaches to each case using the Occupational Injury and Illness Classification System (OIICS).

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC), New York
**Test data** OSHA ITA Case Detail Data, calendar year 2024 (release through 2025-12-31), NAICS 6231
**Source data** US Government work, 17 U.S.C. §105

### Why this test is independent

- Both rules were written on 2025 data and frozen before this run. The 2024 data was never used to write or adjust them.
- The reference labels are OSHA's, not the author's. OSHA states that the OIICS codes in the 2023 and 2024 files were generated in collaboration with the Bureau of Labor Statistics using BLS's automated coder; the file's column names end in `_pred` accordingly. They are an independent reference, not ground truth.

### What is compared

| Rule | OIICS counterpart |
|---|---|
| Violence | Event code 11, "Violent acts by other person" |
| Handling | Event code 71, "Overexertion while moving or manipulating external object(s)", where the source is a person (source code 57, "Person--other than injured or ill worker", or another source naming a patient, resident, or client) |

Code titles are as they appear in the file's `event_title_pred` and `source_title_pred` columns.

OIICS assigns one primary event to each case. The rules can assign more than one label, so a case can be both handling and violence under the rules but only one of them under OIICS. Part of the disagreement below is this difference in counting convention, not rule error.

## 1. Environment

In [1]:
!pip install -q duckdb pandas requests
import os, re, zipfile, hashlib, requests, duckdb, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)

## 2. Download the 2024 case file

OSHA publishes historical files at [osha.gov/itadata](https://www.osha.gov/itadata). OSHA's server returns HTTP 403 without a browser User-Agent header.

The file's SHA-256 checksum and row count are written to `derived/source-file.csv`, so a reader can confirm they have the same file.

In [2]:
URL = 'https://www.osha.gov/sites/default/files/ITA_Case_Detail_Data_2024_through_12-31-2025.zip'
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
ZIP = 'data/ita_2024.zip'
if not os.path.exists(ZIP):
    with requests.get(URL, headers=HDRS, stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(ZIP, 'wb') as f:
            for ch in r.iter_content(1 << 20): f.write(ch)
with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist() if n.lower().endswith('.csv')]
    print('in zip:', names)
    z.extract(names[0], 'data'); CSV = f'data/{names[0]}'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for ch in iter(lambda: f.read(1 << 20), b''): h.update(ch)
    return h.hexdigest()

con = duckdb.connect()
con.execute(f"CREATE TABLE raw AS SELECT * FROM read_csv_auto('{CSV}', all_varchar=true, ignore_errors=true, sample_size=-1)")
cols = [r[0] for r in con.execute('DESCRIBE raw').fetchall()]
n_rows = con.execute('SELECT COUNT(*) FROM raw').fetchone()[0]
src = pd.DataFrame([{'url': URL, 'zip_sha256': sha256(ZIP), 'csv_name': names[0], 'csv_sha256': sha256(CSV), 'rows_all_industries': n_rows}])
src.to_csv('derived/source-file.csv', index=False)
print(f'{n_rows:,} rows, 2024, all industries')
print('OIICS-related columns:', [c for c in cols if re.search(r'event|source|nature|part|oiics', c, re.I)])

in zip: ['ITA_Case_Detail_Data_2024_through_12-31-2025.csv']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

687,695 rows, 2024, all industries
OIICS-related columns: ['nature_code_pred', 'nature_title_pred', 'part_code_pred', 'part_title_pred', 'event_code_pred', 'event_title_pred', 'source_code_pred', 'source_title_pred', 'sec_source_code_pred', 'sec_source_title_pred']


## 3. Nursing-facility cases and their OIICS codes

Column names are resolved from the file, because OSHA has renamed fields between releases.

In [3]:
low = {c.lower(): c for c in cols}
def pick(*cands):
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None
NAICS = pick('naics_code')
NARR = [c for c in [pick('New_nar_before_incident', 'nar_before_incident'), pick('New_nar_what_happened', 'nar_what_happened'),
                    pick('New_nar_injury_illness', 'nar_injury_illness'), pick('New_nar_object_substance', 'nar_object_substance')] if c]
EV_CODE  = next((c for c in cols if re.search(r'event.*code|^event$', c, re.I)), None)
EV_TEXT  = next((c for c in cols if re.search(r'event.*(title|desc|text|narr|keyword)', c, re.I)), None)
SRC_CODE = next((c for c in cols if re.search(r'^source.*code|primary.*source.*code', c, re.I)), None)
SRC_TEXT = next((c for c in cols if re.search(r'source.*(title|desc|text|narr|keyword)', c, re.I)), None)
print('event code:', EV_CODE, '| event text:', EV_TEXT, '| source code:', SRC_CODE, '| source text:', SRC_TEXT)
assert EV_CODE or EV_TEXT, 'No OIICS event column found.'

sel = ', '.join([f'"{c}"' for c in [EV_CODE, EV_TEXT, SRC_CODE, SRC_TEXT] if c])
nar = "lower(concat_ws(' ', " + ', '.join(f'"{c}"' for c in NARR) + '))'
df = con.execute(f'SELECT {sel}, {nar} AS narr FROM raw WHERE "{NAICS}" LIKE \'6231%\'').df()
df = df.rename(columns={EV_CODE: 'ev_code', EV_TEXT: 'ev_text', SRC_CODE: 'src_code', SRC_TEXT: 'src_text'})
for c in ['ev_code', 'ev_text', 'src_code', 'src_text']:
    if c not in df: df[c] = ''
    df[c] = df[c].fillna('').astype(str)
print(f'{len(df):,} nursing-facility cases in 2024')
print('\n== most common OIICS events ==')
print((df.ev_code + '  ' + df.ev_text).value_counts().head(20).to_string())
print('\n== most common OIICS sources ==')
print((df.src_code + '  ' + df.src_text).value_counts().head(15).to_string())

event code: event_code_pred | event text: event_title_pred | source code: source_code_pred | source text: source_title_pred
28,074 nursing-facility cases in 2024

== most common OIICS events ==
55  Exposure to harmful substances                                  7869
71  Overexertion while moving or manipulating external object(s)    6516
43  Slip, trip, stumble or fall on same level                       5059
11  Violent acts by other person                                    2291
66  Contact with non-running objects or equipment                   2103
64  Struck by propelled, falling, or suspended object               1076
73  Bodily position and motion                                       793
61  Contact with other person, nonviolent or intent unknown          707
                                                                     481
53  Exposure to temperature extremes                                 480
65  Struck, caught, or compressed by running powered equipment       215
62 

## 4. Reference labels from OIICS

In [4]:
ev  = (df.ev_code + ' ' + df.ev_text).str.lower()
srcx = (df.src_code + ' ' + df.src_text).str.lower()
PERSON_SRC = srcx.str.contains(r'patient|resident|person|client', regex=True)
# violence: OIICS division 11, intentional injury by person (animals excluded)
df['oiics_violence'] = df.ev_code.str.match(r'^\s*11') | (ev.str.contains(r'violence|assault|hit, kicked|bitten by person|injury by person|intentional injury by', regex=True)
                                                        & ~ev.str.contains(r'animal|insect', regex=True))
# handling: OIICS group 71, overexertion involving outside sources, with a person as the source
df['oiics_handling'] = (df.ev_code.str.match(r'^\s*71') | ev.str.contains(r'overexertion involving outside source|overexertion in (?:lifting|pushing|pulling|holding|carrying|turning|catching)', regex=True)) & PERSON_SRC
print(f"OIICS violence: {int(df.oiics_violence.sum()):,} ({df.oiics_violence.mean():.1%})   "
      f"OIICS handling: {int(df.oiics_handling.sum()):,} ({df.oiics_handling.mean():.1%})")

OIICS violence: 2,291 (8.2%)   OIICS handling: 5,249 (18.7%)


## 5. The frozen rules

Copied unchanged from [`../resident-violence/resident-violence.ipynb`](../resident-violence/resident-violence.ipynb) (violence) and [`../nursing-facility-handling/nursing-facility-handling.ipynb`](../nursing-facility-handling/nursing-facility-handling.ipynb) (handling, strict rule).

In [5]:
PERSON = r"(?:resident|residents|patient|patients|pt|res|client|member|visitor|family member)"
HOSTILE = (r"(?:hit|hits|struck|strikes?|punch\w*|kick\w*|bit|bites?|biting|bitten|scratch\w*|slap\w*|spit|spat|"
           r"head[- ]?butt\w*|pinch\w*|squeez\w*|dug|swung at|wrapped (?:her |his |their )?arms|"
           r"grabb?(?:ed|ing|s)? (?:the )?(?:employee|ee|staff|her|his|my|onto|arms?|forearm|hand|wrist|rt|lt|right|left)\b|"
           r"twist(?:ed|ing) (?:the )?(?:employee|ee|her|his|my)\b(?! (?:right |left )?(?:knee|ankle|back|foot))|push(?:ed|ing) (?:the )?(?:employee|ee|staff|her|him)\b(?! back)|"
           r"spray\w*|assault\w*|attack\w*)")
A = rf"\b{PERSON}\b(?:\W+\w+){{0,5}}?\W+{HOSTILE}"
PASSIVE = rf"\b(?:bit|bitten|hit|struck|kicked|punched|scratched|attacked|assaulted|grabbed)\b(?:\W+\w+){{0,4}}?\W+by (?:a |the )?{PERSON}\b"
AGG = r"\b(?:combative|aggressive|aggression|agitated|assault\w*|attacked|human bite|physical aggression|resident attack)\b"
AGG_ACT = r"\b(?:pull\w*|grab\w*|punch\w*|hit|kick\w*|bit|bite|scratch\w*|twist\w*|push\w*|swung|squeez\w*)\b"
COWORKER = r"\b(?:co-?workers?|employees?|staff members?)\b(?:\W+\w+){0,3}?\W+(?:jumped|attacked|assaulted|punched|sprayed)\b"
ACC = (r"\b(?:hit|struck|bumped|banged)\b (?:her |his |my |their |the )?(?:left |right |l |r )?"
       r"(?:head|hand|knee|arm|elbow|shin|foot|toe|big toe|leg|shoulder|finger)\b(?:\W+\w+){0,3}?\W+(?:on|against|into)\b")

def is_violence(s):
    s = s or ''
    if re.search(COWORKER, s): return True
    if re.search(A, s) or re.search(PASSIVE, s):
        return not (re.search(ACC, s) and not re.search(AGG, s))
    if re.search(AGG, s) and re.search(AGG_ACT, s): return True
    return False

PERSON_H = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE = r"patient handling|resident handling|resident care transfer"
n = df.narr.fillna('')
df['rule_violence'] = n.map(is_violence)
df['rule_handling'] = (n.str.contains(PERSON_H, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)

## 6. Scores

Precision, recall, F1, and Cohen's kappa of each rule against the OIICS reference, and the share of cases each assigns to the category.

In [6]:
def score(pred, truth, name):
    tp = int((pred & truth).sum()); fp = int((pred & ~truth).sum()); fn = int((~pred & truth).sum()); tn = int((~pred & ~truth).sum())
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    po = (tp + tn) / len(pred); pe = ((tp+fp)*(tp+fn) + (fn+tn)*(fp+tn)) / len(pred)**2
    return pd.Series({'rule': name, 'cases': len(pred), 'official positive': int(truth.sum()), 'rule positive': int(pred.sum()),
                      'precision': round(p, 3), 'recall': round(r, 3), 'F1': round(2*p*r/max(p+r, 1e-9), 3),
                      'kappa': round((po - pe) / (1 - pe), 3), 'official share': round(truth.mean(), 3), 'rule share': round(pred.mean(), 3)})
res = pd.DataFrame([score(df.rule_violence, df.oiics_violence, 'violence'),
                    score(df.rule_handling, df.oiics_handling, 'handling')]).set_index('rule')
res.to_csv('derived/rules-vs-oiics-2024.csv')
res.T

rule,violence,handling
cases,28074.000,28074.000
official positive,2291.000,5249.000
rule positive,2202.000,5105.000
precision,0.810,0.695
recall,0.779,0.676
F1,0.794,0.685
kappa,0.776,0.614
official share,0.082,0.187
rule share,0.078,0.182


## 7. Disagreements

Eight cases in each direction, then a random sample of up to 400 disagreements written to `derived/disagreement-sample.csv` with both codes and both predictions. Reading them separates rule errors from differences in counting convention.

The rules are not tuned against these disagreements. Doing so would end the 2024 data's value as an independent test. Any revision will be developed on 2023 data, which also carries OIICS codes, and tested again on 2024.

In [7]:
for name in ['violence', 'handling']:
    r_, o_ = df[f'rule_{name}'], df[f'oiics_{name}']
    for lab, m in [('rule yes, OIICS no', r_ & ~o_), ('rule no, OIICS yes', ~r_ & o_)]:
        s = df[m]
        print(f'\n-- {name}: {lab} ({len(s):,}) --')
        for rr in s.sample(min(8, len(s)), random_state=0).itertuples():
            print(f'  [{rr.ev_code} {rr.ev_text[:40]} | {rr.src_text[:30]}] {rr.narr[:150]}')
disagree = df[(df.rule_violence != df.oiics_violence) | (df.rule_handling != df.oiics_handling)]
(disagree[['ev_code', 'ev_text', 'src_code', 'src_text', 'narr', 'rule_violence', 'oiics_violence', 'rule_handling', 'oiics_handling']]
   .sample(min(400, len(disagree)), random_state=0).to_csv('derived/disagreement-sample.csv', index=False))
print('\nderived:', sorted(os.listdir('derived')))


-- violence: rule yes, OIICS no (418) --
  [71 Overexertion while moving or manipulatin | Person--other than injured or ] assisting residents while unloading from the bus resident in scooter began to come off the ramp. resident cam down crooked and began to fall off scoot
  [64 Struck by propelled, falling, or suspend | Other structural elements] employee was toileting resident in bathroom employee blocked door from hitting resident. pinched left middle finger and left ring finger causing it to
  [66 Contact with non-running objects or equi | Other structural elements] assisting a resident with toileting resident not steady when being transferred to the toilet associate trying to keep resident from falling hit right 
  [61 Contact with other person, nonviolent or | Medical, dental, and surgical ] assisting resident or individual injured worker states injured head picking up trash in resident 's room resident backed up the wheelchair and hit iw.
  [43 Slip, trip, stumble or fall on sam

## Limitations

- **OIICS is itself a coding judgment.** The 2023 and 2024 codes were generated with BLS's automated coder. Agreement measures consistency with OSHA's coding, not with ground truth.
- **One event per case.** OIICS records a primary event; the rules can record more than one. Some disagreement is structural.
- **OSHA sees more than the narrative.** Some cases are coded as violence where the published narrative says little; the coders may have had more information.
- **Missing codes.** 481 of the 28,074 nursing-facility cases carry no event code. They count as neither violence nor handling on the OIICS side.
- **Reporting thresholds.** The comparison inherits ITA's size and industry thresholds; small facilities are under-represented.
- **No establishment is named.**